# Lab 5.1 &mdash; The Supervisor Is a Router

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 2 &middot; Module 5 &mdash; Multi-Agent Collaboration &amp; Orchestration**

### What you'll do
- Build a rule-based supervisor, and measure its routing accuracy honestly
- Read the confusion table &mdash; and notice where every unrecognised request piles up
- Price a misroute: the wasted tokens are everything downstream of the mistake
- Put the model behind the same interface and compare on the same eval set

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Lab 4.2's harness, unchanged.** A supervisor picks a worker the way an agent picks
> a tool, so the measurement is the same one &mdash; which is why this lab comes first.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-5-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 5 labs -- the same payment exceptions, now worked
# by several agents at once, and finally priced against the single agent from Day 1.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

A supervisor decides which specialist handles a request. That is a classification problem with a
known correct answer, so it has an accuracy &mdash; and almost nobody measures it.

It is worth measuring because a misroute is the most expensive mistake in the graph: everything
spent downstream of it answered the wrong question. The supervisor's own call is the cheapest one
in the system, so &ldquo;save money on the router&rdquo; is usually a bad trade.

## Section 1 &mdash; A rule-based supervisor

Keywords, in order, with a default. Unglamorous, free, instant, and identical every time &mdash; and
right far more often than people expect.

In [ ]:
SPECIALISTS = ("ledger", "policy", "sanctions", "writer")

# Order matters: the most specific group goes first.
ROUTING_KEYWORDS = [
    ("sanctions", ("sanction", "embargo", "screening")),
    ("policy",    ("policy", "runbook", "rule", "limit", "breach", "allowed", "permitted")),
    ("writer",    ("draft", "write", "note", "letter", "summar")),
    ("ledger",    ("status", "amount", "record", "look up", "reference", "pull")),
]

def route_by_rule(request: str, default: str = "ledger") -> str:
    """The first keyword group that matches wins. An unrecognised request goes to the default."""
    low = (request or "").lower()
    for specialist, keywords in ROUTING_KEYWORDS:
        if any(k in low for k in keywords):
            return specialist
    return default

In [ ]:
# --- Self-check: Section 1
check("an explicit sanctions request routes to sanctions",
      lambda: route_by_rule("Run the embargo check on ZENITH.") == "sanctions")
check("a policy question routes to policy",
      lambda: route_by_rule("What is the runbook for an INVALID_IBAN return?") == "policy")
check("a lookup routes to the ledger",
      lambda: route_by_rule("What is the status of PMT-1001?") == "ledger")
check("a drafting request routes to the writer",
      lambda: route_by_rule("Draft the customer note for PMT-1002.") == "writer")
check("an unrecognised request goes to the default rather than to a guess",
      lambda: route_by_rule("Tell the client what happened and why.") == "ledger")
check("the default is configurable, because the right default is domain-specific",
      lambda: route_by_rule("nothing matches here", default="writer") == "writer")
check("every rule points at a specialist that exists",
      lambda: all(s in SPECIALISTS for s, _ in ROUTING_KEYWORDS))

## Section 2 &mdash; Measure it

Fifteen requests with a known correct specialist. Four of them state their intent only by
implication &mdash; no keyword names it &mdash; because those are the cases a rule table cannot reach and
the reason anyone reaches for a model.

In [ ]:
ROUTE_EVAL = [
    ("Is PMT-1005 clear of sanctions screening?",                  "sanctions"),
    ("Run the embargo check on ZENITH.",                           "sanctions"),
    ("PMT-1003 breached the limit -- what does policy say?",       "policy"),
    ("What is the runbook for an INVALID_IBAN return?",            "policy"),
    ("Are we allowed to retry this one automatically?",            "policy"),
    ("What is the status of PMT-1001?",                            "ledger"),
    ("Look up the amount on reference PMT-1004.",                  "ledger"),
    ("Pull the record for PMT-1002.",                              "ledger"),
    ("Draft the customer note for PMT-1002.",                      "writer"),
    ("Write up the case summary for the file.",                    "writer"),
    ("Summarise why this payment is held and what happens next.",  "writer"),
    # the four whose intent is implied, not stated
    ("Who is the counterparty on PMT-1003, and is that name a problem?",       "sanctions"),
    ("Is there anything about ZENITH we should worry about before releasing?", "sanctions"),
    ("This one has been sitting for three days. What are we supposed to do?",  "policy"),
    ("Tell the client what happened and why.",                                 "writer"),
]

def selections(router) -> dict:
    """{request: chosen specialist} for the whole eval set."""
    return {request: router(request) for request, _ in ROUTE_EVAL}


def accuracy(sel: dict) -> float:
    """Fraction routed to the expected specialist. No selection counts as wrong."""
    return sum(1 for r, expected in ROUTE_EVAL if sel.get(r) == expected) / len(ROUTE_EVAL)


def confusion(sel: dict) -> dict:
    """{(expected, chosen): count} over the misses -- the pairs whose boundaries overlap."""
    out = {}
    for request, expected in ROUTE_EVAL:
        chosen = sel.get(request)
        if chosen != expected:
            out[(expected, chosen)] = out.get((expected, chosen), 0) + 1
    return out


def _report():
    sel = selections(route_by_rule)
    print(f"rule-based supervisor: {accuracy(sel):.0%} on {len(ROUTE_EVAL)} requests\n")
    for (expected, chosen), n in sorted(confusion(sel).items(), key=lambda kv: -kv[1]):
        print(f"  {n}x  should have been {expected:10} -> went to {chosen}")
guard(_report)

In [ ]:
# --- Self-check: Section 2
_rule = None
def rule_selections():
    global _rule
    if _rule is None:
        _rule = selections(route_by_rule)
    return _rule

check("the eval set covers every specialist",
      lambda: {e for _, e in ROUTE_EVAL} == set(SPECIALISTS))
check("it contains requests whose intent is only implied",
      lambda: sum(1 for r, _ in ROUTE_EVAL
                  if not any(k in r.lower() for _, ks in ROUTING_KEYWORDS for k in ks)) >= 4,
      "an eval set of keyword-shaped requests measures the keywords, not the routing")
check("the rule supervisor gets most of it right",
      lambda: accuracy(rule_selections()) > 0.6)
check("but not all of it -- there is headroom to argue about",
      lambda: accuracy(rule_selections()) < 1.0)
check("every miss lands on the DEFAULT, not on a random specialist",
      lambda: {chosen for _, chosen in confusion(rule_selections())} == {"ledger"},
      "a rule router's failure mode is its default -- that is where unrecognised intent piles up")
check("so the confusion table names one problem, not four",
      lambda: len({chosen for _, chosen in confusion(rule_selections())}) == 1)

## Section 3 &mdash; What a misroute costs

The supervisor's own call is the cheapest thing in the graph. The specialist it wakes up is not.
Price the mistake and the argument about which model to route with settles itself.

In [ ]:
COST = {"supervisor": 120, "ledger": 380, "policy": 420, "sanctions": 90, "writer": 610}

def cost_of(chosen: str) -> int:
    """Tokens spent on one routing decision plus the specialist it woke up."""
    return COST["supervisor"] + COST.get(chosen, 0)


def wasted_tokens(sel: dict) -> int:
    """Tokens spent answering the wrong question."""
    total = 0
    for request, expected in ROUTE_EVAL:
        chosen = sel.get(request)
        if chosen and chosen != expected:
            total += cost_of(chosen)
    return total


def spent_tokens(sel: dict) -> int:
    """Everything the run spent, right or wrong."""
    return sum(cost_of(sel[r]) for r, _ in ROUTE_EVAL if sel.get(r))

In [ ]:
# --- Self-check: Section 3
_perfect = {r: e for r, e in ROUTE_EVAL}

check("a perfect router wastes nothing",
      lambda: wasted_tokens(_perfect) == 0)
check("the waste is the supervisor call plus the specialist it woke up",
      lambda: wasted_tokens({**_perfect,
                             "Tell the client what happened and why.": "ledger"})
              == COST["supervisor"] + COST["ledger"])
check("misrouting to the writer costs more than misrouting to sanctions",
      lambda: cost_of("writer") > cost_of("sanctions"),
      "the cost of a mistake depends on which specialist you woke up, not on the mistake")
check("the rule router wastes a real fraction of what it spends",
      lambda: 0 < wasted_tokens(rule_selections()) < spent_tokens(rule_selections()))
check("cheapening the supervisor cannot recover that waste",
      lambda: wasted_tokens(rule_selections()) > COST["supervisor"] * len(ROUTE_EVAL),
      "even a FREE supervisor would not save what the misroutes already cost -- that is the whole point")

def _price():
    sel = rule_selections()
    spent, wasted = spent_tokens(sel), wasted_tokens(sel)
    print(f"  spent   {spent:>6} tokens")
    print(f"  wasted  {wasted:>6} tokens  ({wasted / spent:.0%} of the bill)")
    print(f"  the supervisor's own calls were only {COST['supervisor'] * len(ROUTE_EVAL)} of that")
guard(_price)

## Run it for real &mdash; the model behind the same interface

Same eval set, same metric, same confusion table. The only thing that changes is the router.

In [ ]:
ROUTE_SYSTEM = ("You route one operations request to exactly one specialist. "
                "Reply with the specialist's name alone -- no punctuation, no explanation.")

SPECIALIST_DESCRIPTIONS = {
    "ledger":    "Reads one payment record: status, amount, counterparty, reason code.",
    "policy":    "Says what the operating policy or runbook requires for a failure reason.",
    "sanctions": "Screens a counterparty name against the watchlist.",
    "writer":    "Turns findings into a summary or a customer-facing note.",
}

def route_with_model(request: str, default: str = "ledger") -> str:
    """Ask the model to pick a specialist. Anything unrecognised falls back to the default."""
    listing = "\n".join(f"- {n}: {d}" for n, d in SPECIALIST_DESCRIPTIONS.items())
    reply = ask(f"Specialists:\n{listing}\n\nRequest: {request}\n\nSpecialist:",
                system=ROUTE_SYSTEM)
    word = (reply or "").strip().strip("`.\"' ").split()
    return word[0] if word and word[0] in SPECIALIST_DESCRIPTIONS else default


if llm_ready():
    def _compare():
        rule = rule_selections()
        model = selections(route_with_model)
        print(f"{'router':16}{'accuracy':>10}{'wasted tokens':>16}")
        print("-" * 44)
        print(f"{'rule-based':16}{accuracy(rule):>9.0%}{wasted_tokens(rule):>16}")
        print(f"{'model':16}{accuracy(model):>9.0%}{wasted_tokens(model):>16}")
        print()
        for (expected, chosen), n in sorted(confusion(model).items(), key=lambda kv: -kv[1]):
            print(f"  model: {n}x  {expected} -> {chosen}")
    guard(_compare)

### Read it

Three things to look at, and the second is the one that decides your design:

1. **Did the model beat 73%?** If not, the rule table is free and deterministic, and you have your
   answer.
2. **Where did the model's misses land?** The rule router's misses all pile up on the default,
   which is one problem you can name. If the model's misses are scattered across four specialists,
   that is four overlapping descriptions &mdash; and Module 4 told you how to fix each one.
3. **Run it twice.** If the same request routes differently on the second run, you have met
   Module 7's opening problem a day early.

The usual production answer is neither: rules for the requests you can name, and a model only for
the ones that fall through &mdash; so you pay for the model on the four cases, not on all fifteen.

In [ ]:
score()

## Your turn

1. Build that hybrid: rules first, model only on a fall-through. Measure its accuracy and its
   cost, and decide whether the saving is worth the second code path.
2. `route_by_rule` returns the first match, so a request mentioning both a policy and a draft goes
   to policy. Make it return every match and let the supervisor dispatch two specialists. What have
   you just committed to paying?
3. Add a fifth outcome to the eval set: `clarify` &mdash; requests where the honest answer is a
   question back to the user. What does that do to your accuracy, and is the drop real?